# Learning-rate tuning, lr0 = 0.005: YOLOv8 invasive plant classification (Group 9)

Only `lr0` changes versus the baseline. Everything else is identical:

| Parameter | Value |
|---|---|
| model | yolov8n-cls.pt |
| epochs | 100 |
| patience | 15 |
| imgsz | 640 |
| batch | 16 |
| **lr0** | **0.005** (baseline 0.01) |
| optimizer | SGD |
| seed | 0 |

Runtime -> Change runtime type -> **T4 GPU**. Outputs are saved as `lr0_0.005*`, so the baseline and the other lr0 runs are never overwritten.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

import torch
assert torch.cuda.is_available(), 'No GPU: switch runtime to T4 GPU (or training will take many hours)'
print(torch.cuda.get_device_name(0))


Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 0.1/112.6 GB disk)
Tesla T4


## 1. Copy training/val/test data onto Colab's local disk (much faster than training from Drive)

In [5]:
import os, shutil

DRIVE_DIR = "/content/drive/MyDrive/IAT360CVProject/classification/dataset"
LOCAL_DIR = "/content/dataset"

shutil.rmtree(LOCAL_DIR, ignore_errors=True)
os.makedirs(LOCAL_DIR)

# Copy the existing splits; use val locally for the Drive valid folder.
for split in ["train", "valid", "test"]:
    target = "val" if split == "valid" else split
    shutil.copytree(f"{DRIVE_DIR}/{split}", f"{LOCAL_DIR}/{target}")

print(os.listdir(LOCAL_DIR))


['test', 'val', 'train']


## 2. Check the local classification folders

In [6]:
# Classification uses species folders instead of data.yaml and box labels.
print(os.listdir("/content/dataset/train"))


['japanese_knotweed', 'giant_hogweed', 'purple_loosestrife', 'cow_parsnip', 'false_solomons_seal', 'lewis_monkeyflower', 'fireweed', 'himalayan_balsam']


## 3. Sanity checks (do not skip: catches wrong classes and missing files)

In [7]:
import glob, os
names = sorted(os.path.basename(p) for p in glob.glob("/content/dataset/train/*") if os.path.isdir(p))
print(names)


['cow_parsnip', 'false_solomons_seal', 'fireweed', 'giant_hogweed', 'himalayan_balsam', 'japanese_knotweed', 'lewis_monkeyflower', 'purple_loosestrife']


In [8]:
import glob, os
from collections import Counter

NAMES = names

for split in ["train", "val", "test"]:
    imgs = glob.glob(f"/content/dataset/{split}/*/*.*")
    imgs_with = Counter(os.path.basename(os.path.dirname(p)) for p in imgs)
    print(f"--- {split}: {len(imgs)} images")
    print("   images per class:", {c: imgs_with[c] for c in NAMES})


--- train: 5999 images
   images per class: {'cow_parsnip': 756, 'false_solomons_seal': 778, 'fireweed': 672, 'giant_hogweed': 735, 'himalayan_balsam': 795, 'japanese_knotweed': 711, 'lewis_monkeyflower': 796, 'purple_loosestrife': 756}
--- val: 1099 images
   images per class: {'cow_parsnip': 140, 'false_solomons_seal': 139, 'fireweed': 132, 'giant_hogweed': 143, 'himalayan_balsam': 141, 'japanese_knotweed': 133, 'lewis_monkeyflower': 143, 'purple_loosestrife': 128}
--- test: 239 images
   images per class: {'cow_parsnip': 34, 'false_solomons_seal': 40, 'fireweed': 15, 'giant_hogweed': 35, 'himalayan_balsam': 25, 'japanese_knotweed': 26, 'lewis_monkeyflower': 40, 'purple_loosestrife': 24}


Expected: train 5,999 images, val 1,099, test 239.
Check that all three splits contain all eight classes. A class with no test images gets no meaningful test score.

## 4. Train the baseline

In [9]:
from ultralytics import YOLO

LR0 = 0.005   # this notebook is the 0.005 run only
RUN = f"lr0_{LR0}"

model = YOLO("yolov8n-cls.pt")   # pretrained ImageNet weights = transfer learning

results = model.train(
    data="/content/dataset",
    epochs=100,
    patience=15,
    imgsz=640,
    batch=16,
    lr0=LR0,
    optimizer="SGD",
    seed=0,
    project="/content/drive/MyDrive/IAT360CVProject/classification/runs",   # saved to Drive so it survives a disconnect
    name=RUN,
    exist_ok=True,
)


Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=lr0_0.005, nbs=64, nms=None, opset=None, optimize=False, optimizer=S

## 5. Evaluate on OUR 239 test photos (the baseline score you report)

In [10]:
from ultralytics import YOLO
best = YOLO(f"/content/drive/MyDrive/IAT360CVProject/classification/runs/{RUN}/weights/best.pt")
metrics = best.val(
    data="/content/dataset",
    split="test",
    project="/content/drive/MyDrive/IAT360CVProject/classification/runs",
    name=f"{RUN}_test",
    exist_ok=True,
    plots=True,
)

print("Top-1 accuracy:", round(metrics.top1, 4))
print("Top-5 accuracy:", round(metrics.top5, 4))


Ultralytics 8.4.207 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,445,128 parameters, 0 gradients, 3.3 GFLOPs
train: /content/dataset/train... found 5999 images in 8/8 classes ✅ 
val: /content/dataset/val... found 1099 images in 8/8 classes ✅ 
test: /content/dataset/test... found 239 images in 8/8 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 129.5±50.8 MB/s, size: 572.3 KB)
test: Scanning /content/dataset/test... 239 images, 0 corrupt: 100% ━━━━━━━━━━━━ 239/239 648.7it/s 0.4s
test: New cache created: /content/dataset/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 15/15 1.3s/it 19.9s
                   all      0.954      0.996
Speed: 0.6ms preprocess, 2.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/drive/.shortcut-targets-by-id/1qUx6swtUgfrnUwlQmDY-ae5z3BzfC4J0/IAT360CVProject/classification/runs/lr0_0.005_test
Top-1 accuracy: 0.954
Top-5 accuracy:

In [ ]:
import glob, random, cv2
import matplotlib.pyplot as plt

random.seed(1)
imgs = random.sample(sorted(glob.glob("/content/dataset/test/*/*.*")), 4)

fig, axes = plt.subplots(len(imgs), 2, figsize=(12, 5 * len(imgs)))
for row, p in enumerate(imgs):
    im = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    gt = os.path.basename(os.path.dirname(p))
    pred = best.predict(p, verbose=False)[0]
    label = pred.names[pred.probs.top1]
    axes[row, 0].imshow(im); axes[row, 0].set_title(f"Your label: {gt}"); axes[row, 0].axis("off")
    axes[row, 1].imshow(im); axes[row, 1].set_title(f"Prediction: {label}"); axes[row, 1].axis("off")
plt.tight_layout(); plt.show()


## 6. Save metrics, example predictions and per-class results

Streams the predictions so Colab does not run out of RAM. Self-contained: works after a kernel restart.

In [1]:
import os, glob, json, shutil
from collections import Counter
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')

LR0 = 0.005
RUN = f"lr0_{LR0}"
RUNS = "/content/drive/MyDrive/IAT360CVProject/classification/runs"
DRIVE_DIR = "/content/drive/MyDrive/IAT360CVProject/classification/dataset"
TEST = "/content/dataset/test"

# only the test split is needed here; re-copy it if local disk was wiped
if not os.path.isdir(TEST):
    os.makedirs("/content/dataset", exist_ok=True)
    shutil.copytree(f"{DRIVE_DIR}/test", TEST)

best = YOLO(f"{RUNS}/{RUN}/weights/best.pt")
paths = sorted(glob.glob(f"{TEST}/*/*.*"))

hit, total = Counter(), Counter()
top1 = top5 = 0
for r in best.predict(paths, stream=True, save=True, project=RUNS,
                      name=f"{RUN}_predictions", exist_ok=True, verbose=False):
    true = os.path.basename(os.path.dirname(r.path))
    pred = r.names[r.probs.top1]
    total[true] += 1
    hit[true] += true == pred
    top1 += true == pred
    top5 += true in [r.names[i] for i in r.probs.top5]

n = len(paths)
out = {
    "run": RUN,
    "hyperparameters": {"model":"yolov8n-cls.pt","epochs":100,"patience":15,"imgsz":640,
                        "batch":16,"lr0":LR0,"optimizer":"SGD","seed":0},
    "test_metrics": {"top1_accuracy": top1 / n, "top5_accuracy": top5 / n},
}
json.dump(out, open(f"{RUNS}/{RUN}_metrics.json", "w"), indent=2)
print(json.dumps(out, indent=2))
for cl in sorted(total):
    print(f"{cl:20s} right class in {hit[cl]}/{total[cl]} photos")
print("Overall:", top1, "/", n)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results saved to /content/drive/.shortcut-targets-by-id/1qUx6swtUgfrnUwlQmDY-ae5z3BzfC4J0/IAT360CVProject/classification/runs/lr0_0.005_predictions
{
  "run": "lr0_0.005",
  "hyperparameters": {
    "model": "yolov8n-cls.pt",
    "epochs": 100,
    "patience": 15,
    "imgsz": 640,
    "batch": 16,
    "lr0": 0.005,
    "optimizer": "SGD",
    "seed": 0
  },
  "test_metrics": {
    "top1_accuracy": 0.9539748953974896,
    "top5_accuracy": 0.99581589958159
  }
}
cow_parsnip          right class in 31/34 photos
false_solomons_seal  right class in 39/40 photos
fireweed             right class in 15/15 photos
giant_hogweed        right class in 32/35 photos
himalayan_balsam     right class in 24/25 photos
japanese_knotweed    right class in 24/26 photos
lewis_monkeyflower   right class in 39/40 photos
purple_loosestrife   right class in 24/24 photos
Overall: 228 

In [ ]:
# (per-class results are printed by the cell above)

## 7. After it finishes

Everything is in `MyDrive/IAT360CVProject/classification/runs/`, named by the `lr0` value:
- `lr0_<value>/weights/best.pt`, `lr0_<value>/results.png`
- `lr0_<value>_test/` (confusion matrix)
- `lr0_<value>_metrics.json`, `lr0_<value>_predictions/`
